## Data Loading and Basic Overview

In [2]:
import pandas as pd

# Load data as a dataframe
traffic_df = pd.read_csv('Data/Metro_Interstate_Traffic_Volume.csv')
traffic_df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [3]:
# Basic info about columns, their datatypes, and how many null values
traffic_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     str    
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  str    
 6   weather_description  48204 non-null  str    
 7   date_time            48204 non-null  str    
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), str(4)
memory usage: 3.3 MB


From this info about the columns, I notice that the holiday column is overwhelmingly full of null values. In the context of the dataset, these null values mean that there was no holiday at the time was data was taken, so its not truly null data. However, a lack of substantial data taken on holdiays means its difficult to adjust a weight for holidays accurately. 

In [4]:
# Statistics about numerical columns
traffic_df.describe().T

,count,mean,std,min,25%,50%,75%,max
temp,48204.0,281.205870,13.338232,0.0,272.16,282.45,291.806,310.07
rain_1h,48204.0,0.334264,44.789133,0.0,0.00,0.00,0.000,9831.30
snow_1h,48204.0,0.000222,0.008168,0.0,0.00,0.00,0.000,0.51
clouds_all,48204.0,49.362231,39.015750,0.0,1.00,64.00,90.000,100.00
traffic_volume,48204.0,3259.818355,1986.860670,0.0,1193.00,3380.00,4933.000,7280.00


In [5]:
# Statistics about categorical columns
traffic_df.describe(include=['str']).T

,count,unique,top,freq
holiday,61,11,Labor Day,7
weather_main,48204,11,Clouds,15164
weather_description,48204,38,sky is clear,11665
date_time,48204,40575,2013-04-18 22:00:00,6


I can tell I will need to change the date_time column into some other format, likely splitting it into month, hour, etc, to create more columns and categories that could be used to determine traffic volume.

## Data Cleaning and Setup

Firstly, lets remove the holiday column, due a lack of representation of holiday times in the dataset. Lets also remove entries that have a holiday in them to remove the holiday's impact on the dataset.

In [6]:
# Find entries with a non-null holiday
indexes = traffic_df.index[~traffic_df['holiday'].isna()]

# Remove those entries alongside the holiday column to ensure holiday has little/no effect on data
clean_df = traffic_df.drop(index=indexes, columns=['holiday'])
clean_df.reset_index(drop=True, inplace=True)
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48143 entries, 0 to 48142
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   temp                 48143 non-null  float64
 1   rain_1h              48143 non-null  float64
 2   snow_1h              48143 non-null  float64
 3   clouds_all           48143 non-null  int64  
 4   weather_main         48143 non-null  str    
 5   weather_description  48143 non-null  str    
 6   date_time            48143 non-null  str    
 7   traffic_volume       48143 non-null  int64  
dtypes: float64(3), int64(2), str(3)
memory usage: 2.9 MB


Next, lets convert the date_time columns values from strings to datetimes.

In [7]:
# Convert column from string to datetime datatype
clean_df['date_time'] = pd.to_datetime(clean_df['date_time'])
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48143 entries, 0 to 48142
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   temp                 48143 non-null  float64       
 1   rain_1h              48143 non-null  float64       
 2   snow_1h              48143 non-null  float64       
 3   clouds_all           48143 non-null  int64         
 4   weather_main         48143 non-null  str           
 5   weather_description  48143 non-null  str           
 6   date_time            48143 non-null  datetime64[us]
 7   traffic_volume       48143 non-null  int64         
dtypes: datetime64[us](1), float64(3), int64(2), str(2)
memory usage: 2.9 MB


Lets also extract the hour from date_time, as it is an intuitive factor in traffic volume.

In [8]:
# Extract hour from date_time
clean_df['hour'] = clean_df['date_time'].dt.hour

# Drop the date_time column as we have the relevant information from it
clean_df = clean_df.drop(columns=['date_time'])
clean_df.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,traffic_volume,hour
0,288.28,0.0,0.0,40,Clouds,scattered clouds,5545,9
1,289.36,0.0,0.0,75,Clouds,broken clouds,4516,10
2,289.58,0.0,0.0,90,Clouds,overcast clouds,4767,11
3,290.13,0.0,0.0,90,Clouds,overcast clouds,5026,12
4,291.14,0.0,0.0,75,Clouds,broken clouds,4918,13


## Feature Engineering

Now lets prep our data for learning using methods like one-hot encoding and min-max scaling

In [9]:
# First, lets define features and the label
features_df = clean_df[['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description','hour']]
label_df = clean_df[['traffic_volume']]

display(features_df.head())
display(label_df.head())

,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,hour
0,288.28,0.0,0.0,40,Clouds,scattered clouds,9
1,289.36,0.0,0.0,75,Clouds,broken clouds,10
2,289.58,0.0,0.0,90,Clouds,overcast clouds,11
3,290.13,0.0,0.0,90,Clouds,overcast clouds,12
4,291.14,0.0,0.0,75,Clouds,broken clouds,13


,traffic_volume
0,5545
1,4516
2,4767
3,5026
4,4918


In [10]:
from sklearn.preprocessing import OneHotEncoder

# Now lets one-hot encode our categorical columns
for column_name in ['weather_main', 'weather_description', 'hour']:
    column = features_df[[column_name]]

    column_encoder = OneHotEncoder(sparse_output=False)
    encoded_df = pd.DataFrame(column_encoder.fit_transform(column), columns=column_encoder.get_feature_names_out(),
                              index=features_df.index)

    features_df = pd.concat([features_df, encoded_df], axis=1)
    features_df.drop(columns=[column_name], inplace=True)

features_df.head()
    

,temp,rain_1h,snow_1h,clouds_all,weather_main_Clear,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,288.28,0.0,0.0,40,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,289.36,0.0,0.0,75,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,289.58,0.0,0.0,90,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,290.13,0.0,0.0,90,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,291.14,0.0,0.0,75,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Now lets min-max scale our continious numerical values
column_names = ['temp', 'rain_1h', 'snow_1h', 'clouds_all']
scaler = MinMaxScaler()
features_df[column_names] = scaler.fit_transform(features_df[column_names])

features_df.head()


,temp,rain_1h,snow_1h,clouds_all,weather_main_Clear,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,0.929726,0.0,0.0,0.40,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.933209,0.0,0.0,0.75,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.933918,0.0,0.0,0.90,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.935692,0.0,0.0,0.90,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.938949,0.0,0.0,0.75,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
